# All sessions — the random timeout / banishment task

Performance **across sessions and mice**. The unit is a **session** (dot), aggregated to an **animal**
(diamond / mean) — never the individual trial. Trials appear once, as counts. Occupancy and heading —
which can't be a single dot — are shown as **exemplar sessions** (the most- vs least-significant session
per mouse), not an average.

**Caching:** tables + the per-session maps/curves are saved once. `REBUILD=False` loads them (no logs
read); `REBUILD=True` re-reads the logs.

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore', message='.*All-NaN slice.*')

_HERE = Path.cwd()
_MP = None
for _b in (_HERE, *_HERE.parents):
    if (_b / 'mixed_perf.py').exists(): _MP = _b; break
    if (_b / 'mixed_protocol' / 'mixed_perf.py').exists(): _MP = _b / 'mixed_protocol'; break
assert _MP is not None, f'cannot locate mixed_perf.py from {_HERE}'
sys.path.insert(0, str(_MP))
import importlib, mixed_perf as mp
importlib.reload(mp)          # reload so a git pull is picked up WITHOUT restarting the kernel
print('mixed_perf loaded from', mp.__file__)
assert hasattr(mp, 'collected_joyfine_curves_time'), \
    'stale mixed_perf.py -- git pull, then Kernel > Restart and run from cell 1'

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
MAIN_DIR = '/path/to/MAIN_DIR'      # <-- root folder that holds the MOUSE folders  (MAIN_DIR/<mouse>/<session>/log.json)
VIEW_SCALE = mp.DEFAULT_VIEW_SCALE   # world zoom (not in the log; 0.35 default)
PATTERN = '*/*/log.json'             # <mouse>/<session>/log.json ; falls back to a recursive search
REBUILD = False                      # False = load saved tables if present ; True = re-read every log

MAIN_DIR = Path(MAIN_DIR); assert MAIN_DIR.exists(), f'MAIN_DIR does not exist: {MAIN_DIR}'
CACHE = MAIN_DIR / 'mixed_protocol_df'

### option — save figures as 600-dpi PNGs

Set `SAVE_PNG = True` below, then run the cells you want saved. Files go to `PNG_DIR`.

In [ ]:
# ── OPTION: save every figure as a high-quality PNG ──────────────────────────────
# Set SAVE_PNG = True, then run any cell -> its figure(s) are ALSO written to PNG_DIR at PNG_DPI.
# Leave False for normal interactive use. Files are numbered + named from the figure title.
import re
SAVE_PNG = False
PNG_DPI  = 600
PNG_DIR  = MAIN_DIR / 'figures'
_fig_n, _seen_ids = [0], set()
if not getattr(plt, '_save_png_hooked', False):
    plt._orig_show = plt.show
    def _save_show(*a, **k):
        if SAVE_PNG:
            PNG_DIR.mkdir(parents=True, exist_ok=True)
            for _num in plt.get_fignums():
                _f = plt.figure(_num)
                if id(_f) in _seen_ids:
                    continue
                _st = getattr(_f, '_suptitle', None)
                _t = _st.get_text() if _st else (_f.axes[0].get_title() if _f.axes and _f.axes[0].get_title() else '')
                _name = re.sub(r'[^\w \-]', '', _t).strip().replace(' ', '_')[:50] or 'figure'
                _fig_n[0] += 1
                _f.savefig(PNG_DIR / f'{{_fig_n[0]:02d}}_{{_name}}.png', dpi=PNG_DPI, bbox_inches='tight')
                _seen_ids.add(id(_f))
        return plt._orig_show(*a, **k)
    plt.show = _save_show
    plt._save_png_hooked = True
print(f'SAVE_PNG = {{SAVE_PNG}}  ->  set True to write {{PNG_DPI}}-dpi PNGs into {{PNG_DIR}}')

## 1 — load or build the tables

`REBUILD=False` loads the cache in a second. Otherwise every log is read once (progress bars) and
saved: **ALL** (per collection), **SUM** (per **session**), **PS** (per **session×effect**), plus the
per-session occupancy maps and heading curves used by the exemplar plots.

In [ ]:
_files = ['all.pkl', 'summary.pkl', 'per_effect.pkl', 'occ_sess.pkl', 'head_sess.pkl', 'joy_sess.pkl']
_missing = [f for f in _files if not (CACHE / f).exists()]
_have = CACHE.exists() and not _missing            # REBUILD is the ONLY switch; only MISSING FILES force a build
# newer analysis columns; if the cache predates them it still LOADS -- just refresh once (REBUILD=True) to use them
_NEWCOLS = {'gen_banish', 'gen_timeout', 'trial_banish', 'trial_timeout', 'col_banish', 'col_timeout', 'p_timeout', 'p_banish', 'life_banish_med', 'life_timeout_med'}
print(f'cache: {CACHE}')
print(f'  REBUILD={REBUILD} | cache files present {len(_files) - len(_missing)}/{len(_files)}'
      + (f' | MISSING: {_missing}' if _missing else ''))

if not REBUILD and _have:
    ALL = pd.read_pickle(CACHE / 'all.pkl'); SUM = pd.read_pickle(CACHE / 'summary.pkl')
    PS = pd.read_pickle(CACHE / 'per_effect.pkl')
    OCC_SESS = pickle.load(open(CACHE / 'occ_sess.pkl', 'rb')); HEAD_SESS = pickle.load(open(CACHE / 'head_sess.pkl', 'rb'))
    JOY_SESS = pickle.load(open(CACHE / 'joy_sess.pkl', 'rb'))
    _absent = _NEWCOLS - set(SUM.columns)
    if _absent:
        print(f'==> LOADED from cache (no logs read). NOTE: this cache predates {sorted(_absent)} -- '
              f'those sections will show a hint; set REBUILD=True ONCE to refresh them.')
    else:
        print('==> LOADED from cache (no logs read).')
else:
    _reason = ('REBUILD=True' if REBUILD else
               (f'cache incomplete, missing files {_missing}' if _missing else 'no cache yet'))
    print(f'==> BUILDING (reason: {_reason}); reading the logs...')
    found = mp.find_sessions(MAIN_DIR, PATTERN)
    SESSIONS = []
    for p, log in mp._progress(found, 'building session tables'):
        mouse, session, mouse_folder = mp.session_label(p, log, main_dir=MAIN_DIR)
        df = mp.build_session_df(log, view_scale=VIEW_SCALE, session=session, mouse=mouse)
        df['mouse_folder'] = mouse_folder; df['date'] = str(log.get('experiment_data', {}).get('datetime', ''))[:19]
        SESSIONS.append((mouse, session, log, df))
    assert len(SESSIONS), 'no mixed-protocol sessions found under MAIN_DIR (check the path / PATTERN / layout)'
    ALL = pd.concat([df for _, _, _, df in SESSIONS], ignore_index=True)
    ALL['head_deg'] = np.degrees(np.arccos(ALL['heading_align'].clip(-1, 1)))
    SUM = pd.DataFrame([{**mp.session_summary(log, df), 'date': df['date'].iloc[0],
                         'mouse_folder': df['mouse_folder'].iloc[0]} for _, _, log, df in SESSIONS])
    PS = (ALL.groupby(['mouse', 'session', 'effect'])
             .agg(n=('idx', 'size'), time_s=('dt_prev_ms', lambda v: float(np.nanmean(v)) / 1000),
                  dist=('dist_prev', 'mean'), pe=('path_efficiency', 'median'), head_deg=('head_deg', 'median'))
             .reset_index())
    HALF, BINS, GRID = 900, 45, np.linspace(-8, 0, 60)      # per-session maps + heading curves (one pass)
    OCC_SESS = {'half': HALF, 'grid': GRID, **{e: {} for e, _ in mp.TYPES}}
    HEAD_SESS = {'grid': GRID, **{e: {} for e, _ in mp.TYPES}}
    JOY_SESS = {'grid': GRID, **{e: {} for e, _ in mp.TYPES}}
    for mouse, session, log, df in mp._progress(SESSIONS, 'occupancy + heading + joystick'):
        for e, _ in mp.TYPES:
            ox, oy, dt = mp.collection_offsets(log, df, e, window_s=3.0)
            if ox.size >= 5:
                Hh, _, _ = np.histogram2d(ox, oy, bins=BINS, range=[[-HALF, HALF], [-HALF, HALF]], weights=dt)
                if Hh.sum() > 0: OCC_SESS[e][(mouse, session)] = Hh / Hh.sum()
            curves = mp.collected_curves_time(log, df, e, window_s=8.0)
            if curves:
                M = np.vstack([np.interp(GRID, t, er, left=np.nan, right=np.nan) for t, er in curves])
                HEAD_SESS[e][(mouse, session)] = np.nanmedian(M, axis=0)
            jcur = mp.collected_joyfine_curves_time(log, df, e, window_s=8.0)
            if jcur:
                Mj = np.vstack([np.interp(GRID, t, v, left=np.nan, right=np.nan) for t, v in jcur])
                JOY_SESS[e][(mouse, session)] = np.nanmedian(Mj, axis=0)
    CACHE.mkdir(exist_ok=True)
    ALL.to_pickle(CACHE / 'all.pkl'); SUM.to_pickle(CACHE / 'summary.pkl'); PS.to_pickle(CACHE / 'per_effect.pkl')
    pickle.dump(OCC_SESS, open(CACHE / 'occ_sess.pkl', 'wb')); pickle.dump(HEAD_SESS, open(CACHE / 'head_sess.pkl', 'wb'))
    pickle.dump(JOY_SESS, open(CACHE / 'joy_sess.pkl', 'wb'))
    _absent = _NEWCOLS - set(SUM.columns)
    if _absent:
        print(f'\n⚠️  built SUM is missing {sorted(_absent)} -- your mixed_perf.py is likely stale in this '
              f'kernel. Re-run the import cell (it reloads) or restart the kernel, then rebuild once.')
    print(f'==> BUILT and SAVED -> {CACHE}   (re-run with REBUILD=False to LOAD instantly next time)')
print(f'{SUM.shape[0]} sessions | {SUM.mouse.nunique()} mice')

## 2 — inventory + shared style

In [ ]:
inv = SUM[['mouse', 'mouse_folder', 'session', 'date', 'n_coll', 'n_reward', 'n_timeout',
           'n_banish', 'n_escape', 'p_all']].sort_values(['mouse', 'date']).reset_index(drop=True)
mice = sorted(SUM['mouse'].unique())
# mouse palette: soft, colour-blind-safe (Set2 then Set3 for overflow); no pure red (red = the p=0.05 line)
_mpal = [*plt.cm.Set2(np.linspace(0, 1, 8)), *plt.cm.Set3(np.linspace(0, 1, 12))]
MCOL = {m: _mpal[i % len(_mpal)] for i, m in enumerate(mice)}
from matplotlib.lines import Line2D
_mlegend = [Line2D([0], [0], marker='o', ls='', color=MCOL[m], mec='k', label=m) for m in mice]
_rng = np.random.default_rng(0)

def dots_by_mouse(ax, data, key, ylabel, ylim=None):
    '''dot = one SESSION (colour = mouse) at that mouse's x; diamond = the ANIMAL mean.'''
    for i, m in enumerate(mice):
        v = data[data.mouse == m][key].dropna().values
        ax.scatter(np.full(len(v), i) + _rng.uniform(-.11, .11, len(v)), v, s=42, color=MCOL[m], edgecolor='k', lw=.4, alpha=.8, zorder=2)
        if len(v): ax.scatter([i], [np.nanmean(v)], marker='D', s=95, color=MCOL[m], edgecolor='k', lw=1.3, zorder=3)
    ax.set_xticks(range(len(mice))); ax.set_xticklabels(mice, rotation=30, fontsize=7)
    ax.set_ylabel(ylabel, fontsize=8)
    if ylim: ax.set_ylim(*ylim)

def dots_by_cat(ax, data, catcol, order, key, ylabel, labels=None, ylim=None):
    '''dot = one SESSION (colour = mouse) grouped by a category; black bar = across-session mean.'''
    for i, cat in enumerate(order):
        sub = data[data[catcol] == cat]
        for m in mice:
            v = sub[sub.mouse == m][key].dropna().values
            ax.scatter(np.full(len(v), i) + _rng.uniform(-.12, .12, len(v)), v, s=36, color=MCOL[m], edgecolor='k', lw=.3, alpha=.8, zorder=2)
        allv = sub[key].dropna().values
        if len(allv): ax.scatter([i], [np.nanmean(allv)], marker='_', s=600, color='k', zorder=4)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(labels or order)
    ax.set_ylabel(ylabel, fontsize=8)
    if ylim: ax.set_ylim(*ylim)

# significance: a session is significant if it beats chance on ANY negative icon --
# pooled (p_all), timeout-only (p_timeout) OR banish-only (p_banish). p_all alone POOLS the two
# negative icons and hides an animal that avoids one but not the other, so 'any' is the default.
SIG_THRESH = 0.05
SIG_MODE = 'p_all'        # significance is p_all < 0.05 (change to 'p_timeout'/'p_banish', or 'any' for any-of-three)
_PCOLS = ['p_all', 'p_timeout', 'p_banish']
SUM['p_min'] = SUM[_PCOLS].min(axis=1)
if SIG_MODE == 'any':
    SUM['sig'] = SUM[_PCOLS].lt(SIG_THRESH).any(axis=1); SIG_COL = 'any(p_all|p_timeout|p_banish)'
else:
    SIG_COL = SIG_MODE; SUM['sig'] = SUM[SIG_COL] < SIG_THRESH
SUM['sig_which'] = [','.join(c[2:] for c in _PCOLS if r[c] < SIG_THRESH) or '-' for _, r in SUM.iterrows()]
_sigmap = SUM.set_index(['mouse', 'session'])['sig'].to_dict()
PS['sig'] = [_sigmap.get((m, s), False) for m, s in zip(PS.mouse, PS.session)]
# show which negative icon drove each session in the inventory table
inv = inv.merge(SUM[['mouse', 'session', 'p_timeout', 'p_banish', 'sig_which']], on=['mouse', 'session'], how='left')

def dots_by_sig(ax, data, key, ylabel, ylim=None):
    dots_by_cat(ax, data, 'sig', [True, False], key, ylabel, labels=['significant', 'non-signif.'], ylim=ylim)

def exemplars(mouse):
    '''(most, least) significant session rows for a mouse = lowest / highest p_all.'''
    d = SUM[(SUM.mouse == mouse)].dropna(subset=['p_all']).sort_values('p_all')
    if len(d) == 0: return None, None
    return d.iloc[0], d.iloc[-1]

print('dots = sessions (colour = mouse), diamond/bar = mean')
inv

## 2b — stochastic p across sessions (line per mouse)

Each mouse's stochastic p over its sessions, ordered in time. **p_all** (vs all punishments) is the
headline; p_timeout and p_banish are shown alongside. A point **below the red 0.05 line** is a
significant session. This is the trajectory view — does discrimination sharpen (p fall) with training?

In [ ]:
import matplotlib.dates as mdates
_sortcol = 'date' if 'date' in SUM.columns else 'session'
_po = SUM.sort_values(['mouse', _sortcol]).copy()
_po['_dt'] = pd.to_datetime(_po['date'], errors='coerce') if 'date' in _po.columns else pd.NaT
_use_dates = _po['_dt'].notna().any()

def _xy(d):
    d = d.sort_values('_dt') if _use_dates else d
    return d, (d['_dt'].values if _use_dates else np.arange(len(d)))

def _datefmt(a):
    if _use_dates:
        a.xaxis.set_major_locator(mdates.AutoDateLocator())
        a.xaxis.set_major_formatter(mdates.ConciseDateFormatter(a.xaxis.get_major_locator()))
    a.set_xlabel('date' if _use_dates else 'session (ordered by date)')
    plt.setp(a.get_xticklabels(), rotation=45, ha='right', fontsize=7)

_ptitlecol = {'p_all': '0.15', 'p_timeout': mp.COLR['timeout'], 'p_banish': mp.COLR['banish']}
pgrid = [('p_all', 'p (vs ALL punishments)'), ('p_timeout', 'p (vs timeout)'), ('p_banish', 'p (vs banishment)')]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for a, (pk, lbl) in zip(axes, pgrid):
    for m in mice:
        d, x = _xy(_po[_po.mouse == m])
        if len(d):
            a.plot(x, d[pk].values, '-o', color=MCOL[m], ms=5, lw=1.9, mec='0.3', mew=0.5, label=m)
    a.axhline(0.05, ls='--', color='r', lw=1); a.set_ylim(-0.03, 1.03)
    a.set_title(lbl, fontsize=10, color=_ptitlecol[pk], fontweight='bold')
    a.text(0.02, 0.07, 'significant', color='r', fontsize=7, transform=a.transAxes)
    _datefmt(a)
axes[0].set_ylabel('stochastic p')
axes[0].legend(handles=_mlegend, fontsize=6, title='mouse', loc='upper right')
plt.suptitle('stochastic p-value across sessions — one line per mouse (below the red line = significant)')
plt.tight_layout(); plt.show()

# p_all alone, larger, markers coloured by significance
fig, ax = plt.subplots(figsize=(11, 4.4))
for m in mice:
    d, x = _xy(_po[_po.mouse == m])
    if len(d):
        x = np.asarray(x)
        ax.plot(x, d.p_all.values, '-', color=MCOL[m], lw=1.8, label=m, zorder=2)
        sig = d.p_all.values < SIG_THRESH
        ax.scatter(x[sig], d.p_all.values[sig], s=70, facecolor=MCOL[m], edgecolor='k', lw=1.1, zorder=3)
        ax.scatter(x[~sig], d.p_all.values[~sig], s=45, facecolor='white', edgecolor=MCOL[m], lw=1.3, zorder=3)
ax.axhline(0.05, ls='--', color='r', lw=1); ax.set_ylim(-0.03, 1.03)
ax.set_ylabel('p_all (vs all punishments)')
ax.set_title('p_all across sessions per mouse  (filled = significant, open = not)')
ax.legend(handles=_mlegend, fontsize=6, title='mouse', loc='upper right')
_datefmt(ax)
plt.tight_layout(); plt.show()

## 3 — counts (the only trial-level view)

Three pies: how the collections split by effect, how the rewards split by multiplier (combo) level, and
how the sessions split across mice.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
effs = ['single_reward', 'timeout', 'banish', 'unbanish']
tot = [int((ALL.effect == e).sum()) for e in effs]
ax[0].pie(tot, labels=[f'{mp.ELABEL[e]}\n{v}' for e, v in zip(effs, tot)],
          colors=[mp.COLR[e] for e in effs], autopct='%1.0f%%', startangle=90)
ax[0].set_title(f'collections by effect  (n={sum(tot)})')

mult = ALL.loc[ALL.valence == 'positive', 'multiplier'].dropna().astype(int)
mc = mult.value_counts().sort_index()
ax[1].pie(mc.values, labels=[f'x{k}\n{v}' for k, v in mc.items()],
          colors=plt.cm.Greens(np.linspace(0.4, 0.9, len(mc))), autopct='%1.0f%%', startangle=90)
ax[1].set_title(f'rewards by multiplier level  (n={len(mult)})')

spm = inv.groupby('mouse').size().reindex(mice)
ax[2].pie(spm.values, labels=[f'{m}\n{int(v)}' for m, v in spm.items()],
          colors=[MCOL[m] for m in mice], autopct='%1.0f%%', startangle=90)
ax[2].set_title(f'sessions per mouse  ({SUM.shape[0]} sessions, {len(mice)} mice)')
plt.tight_layout(); plt.show()

In [ ]:
# per mouse: AVERAGE collections per session by effect (session = the unit, NOT pooled across sessions)
effs4 = ['single_reward', 'timeout', 'banish', 'unbanish']
ncols4 = {'single_reward': 'n_reward', 'timeout': 'n_timeout', 'banish': 'n_banish', 'unbanish': 'n_escape'}
fig, ax = plt.subplots(1, len(mice), figsize=(4 * len(mice), 4), squeeze=False)
for a, m in zip(ax[0], mice):
    sm = SUM[SUM.mouse == m]
    v = [float(sm[ncols4[e]].mean()) for e in effs4]
    if sum(v) > 0:
        a.pie(v, labels=[f'{mp.ELABEL[e]}\n{c:.1f}' for e, c in zip(effs4, v)],
              colors=[mp.COLR[e] for e in effs4], autopct='%1.0f%%', startangle=90)
    a.set_title(f'{m}  ({len(sm)} sessions, mean/session)', fontsize=9)
plt.suptitle('AVERAGE collections per session by effect, per mouse (each session weighted equally, not pooled)')
plt.tight_layout(); plt.show()

### 3b — punishments generated: random or biased?

Each spawn puts ONE punishment on the board that is randomly a **banishment** or a **timeout**. Here:
how many of each were **generated** (spawned, not just collected), the timeout fraction, and whether
the banish/timeout draw looks random — a binomial balance test (fraction vs 50/50) and a runs test on
the B/T order (random draw vs a balanced/streaky pseudo-random generator). Averaged per mouse.

In [ ]:
if 'trial_banish' not in SUM.columns:
    print('on-board/collected columns absent -> the cache predates them. Set REBUILD=True and re-run from section 1.')
else:
    print('per mouse, summed over its sessions        on-board (trials) -> collected')
    for m in mice:
        sm = SUM[SUM.mouse == m]
        print(f'  {m:12s}  banishment {sm.trial_banish.sum():4.0f} -> {sm.col_banish.sum():4.0f}     '
              f'timeout {sm.trial_timeout.sum():4.0f} -> {sm.col_timeout.sum():4.0f}')

    # on-board (spawn-batch trials it was present) vs collected, PER MOUSE, one panel per punishment type
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.3)); xx = np.arange(len(mice)); w = 0.38
    for a, (e, oc, cc, lbl) in zip(ax, [('banish', 'trial_banish', 'col_banish', 'banishment'),
                                        ('timeout', 'trial_timeout', 'col_timeout', 'timeout')]):
        ov = [SUM[SUM.mouse == m][oc].mean() for m in mice]
        cv = [SUM[SUM.mouse == m][cc].mean() for m in mice]
        a.bar(xx - w/2, ov, w, color=mp.COLR[e], label='on-board (trials)')
        a.bar(xx + w/2, cv, w, color=mp.COLR[e], alpha=0.4, hatch='//', label='collected')
        for i in range(len(mice)):
            a.text(xx[i] - w/2, ov[i], f'{ov[i]:.1f}', ha='center', va='bottom', fontsize=7)
            a.text(xx[i] + w/2, cv[i], f'{cv[i]:.1f}', ha='center', va='bottom', fontsize=7)
        a.set_xticks(xx); a.set_xticklabels(mice, rotation=30, fontsize=7)
        a.set_ylabel('mean per session'); a.set_title(lbl + ' (mean/session)'); a.legend(fontsize=7)
    plt.suptitle('punishments on-board (trials present) vs collected, per mouse (mean per session — not pooled)')
    plt.tight_layout(); plt.show()

### 3c — how long each negative icon lingers before it is hit

For every punishment icon, how many **trials (spawn batches)** it stayed on screen before it was
collected. **Higher = the animal takes more trials to reach it = avoided longer.** The distribution is
skewed (most negative icons are hit within a trial or two, a few linger), so both the median (robust) and the
mean are shown, per mouse.

In [ ]:
if 'life_banish_med' not in SUM.columns:
    print('lifetime columns absent -> the cache predates them. Set REBUILD=True and re-run from section 1.')
else:
    xx = np.arange(len(mice)); w = 0.38
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for a, (stat, tag) in zip(ax, [('med', 'median'), ('mean', 'mean')]):
        a.bar(xx - w/2, [SUM[SUM.mouse == m][f'life_banish_{stat}'].mean() for m in mice], w, color=mp.COLR['banish'], label='banishment')
        a.bar(xx + w/2, [SUM[SUM.mouse == m][f'life_timeout_{stat}'].mean() for m in mice], w, color=mp.COLR['timeout'], label='timeout')
        a.set_xticks(xx); a.set_xticklabels(mice, rotation=30, fontsize=7)
        a.set_ylabel(f'{tag} trials on screen / negative icon'); a.set_title(f'{tag} trials until hit'); a.legend(fontsize=7)
    plt.suptitle('how long each negative icon lingers before it is hit (higher = reached more slowly / avoided longer)')
    plt.tight_layout(); plt.show()
    _lb, _lt = SUM.life_banish_med.mean(), SUM.life_timeout_med.mean()
    print(f'across mice (mean of per-session medians): banishment {_lb:.2f} vs timeout {_lt:.2f} trials on screen -> '
          + ('banishment lingers longer (avoided longer)' if _lb > _lt else 'timeout lingers longer (avoided longer)'))

## 4 — choice & avoidance (session = dot, animal = diamond)

**Reward rate** = rewards / (rewards + negatives) per session, where **rewards = the COUNT of `single_reward` collections** (each collection counts once, regardless of its multiplier — it is NOT the droplet/drops total; the multiplier is shown separately). **Stochastic p** = binomial P(≥ this many
rewards if each good/bad choice were 'good' at the world ratio 0.667); the three versions differ only in
what counts as *bad* — all negative icons, timeout only, banishment only — and the red line is p = 0.05.

In [ ]:
grid = [('reward_rate', 'reward rate', (0, 1.02)), ('p_all', 'p (vs all negatives)', (0, 1.02)),
        ('p_timeout', 'p (vs timeout)', (0, 1.02)), ('p_banish', 'p (vs banishment)', (0, 1.02)),
        ('mult_mean', 'mean multiplier', None)]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for a, (key, lbl, ylim) in zip(axes, grid):
    dots_by_mouse(a, SUM, key, lbl, ylim=ylim); a.set_title(lbl, fontsize=9)
    if key.startswith('p_'):
        a.axhline(0.05, ls='--', color='r', lw=.9)
        # sample size behind each p: mean n_reward vs mean n of the relevant negative icon, per mouse
        _hz = {'p_all': None, 'p_timeout': 'n_timeout', 'p_banish': 'n_banish'}[key]
        _lab = {'p_all': 'neg', 'p_timeout': 'TO', 'p_banish': 'ban'}[key]
        for i, m in enumerate(mice):
            sm = SUM[SUM.mouse == m]
            nr = sm.n_reward.mean(); nn = (sm.n_timeout + sm.n_banish).mean() if _hz is None else sm[_hz].mean()
            a.text(i, 1.0, f'R{nr:.0f}\n{_lab}{nn:.0f}', ha='center', va='top', fontsize=6, color='0.35')
axes[0].legend(handles=_mlegend, fontsize=6, title='mouse', loc='lower left')
plt.suptitle('choice & avoidance — dot = session, diamond = animal mean  (R = mean rewards, TO/ban/neg = mean negative icon n, per mouse)')
plt.tight_layout(); plt.show()

In [ ]:
# significant sessions per mouse, BROKEN DOWN by which p-value (not mixed together)
pkeys = [('p_all', 'vs all neg', '#666666'), ('p_timeout', 'vs timeout', mp.COLR['timeout']), ('p_banish', 'vs banish', mp.COLR['banish'])]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6)); xx = np.arange(len(mice)); w = 0.26
for k, (pk, lbl, c) in enumerate(pkeys):
    counts = [int((SUM[SUM.mouse == m][pk] < SIG_THRESH).sum()) for m in mice]
    ax[0].bar(xx + (k - 1) * w, counts, w, color=c, label=lbl)
    for i, v in enumerate(counts):
        if v: ax[0].text(xx[i] + (k - 1) * w, v + 0.03, str(v), ha='center', fontsize=7)
ax[0].set_xticks(xx); ax[0].set_xticklabels(mice, rotation=30, fontsize=7)
ax[0].set_ylabel('significant sessions (of that mouse)'); ax[0].legend(fontsize=7, title='p < 0.05 on')
ax[0].set_title(f'significant sessions per mouse, by which p (thresh {SIG_THRESH})')
# right: the mean n behind each p, so a p resting on few negative icons is visible
for k, (col, lbl, c) in enumerate([('n_reward', 'reward', '#009E73'), ('n_timeout', 'timeout', mp.COLR['timeout']), ('n_banish', 'banishment', mp.COLR['banish'])]):
    ax[1].bar(xx + (k - 1) * w, [SUM[SUM.mouse == m][col].mean() for m in mice], w, color=c, label=lbl)
ax[1].set_xticks(xx); ax[1].set_xticklabels(mice, rotation=30, fontsize=7)
ax[1].set_ylabel('mean count / session'); ax[1].legend(fontsize=7)
ax[1].set_title('mean collections per session (the n behind each p)')
plt.tight_layout(); plt.show()
print('reminder: p_timeout / p_banish rest on how many of THAT negative icon occurred -- e.g. only ~5 timeouts in ~51 collections makes the timeout test weak, so read the n alongside the p.')

### 4d — how many sessions are significant (pie)

In [ ]:
# significant vs non-significant sessions: overall + per mouse
_sc = ['#2ca25f', '#bdbdbd']
fig, ax = plt.subplots(1, len(mice) + 1, figsize=(3.4 * (len(mice) + 1), 3.6), squeeze=False)
ns = int(SUM.sig.sum()); nn = int((~SUM.sig).sum())
ax[0, 0].pie([ns, nn], labels=[f'significant\n{ns}', f'non-sig\n{nn}'], colors=_sc, autopct='%1.0f%%', startangle=90)
ax[0, 0].set_title(f'ALL mice  (n={len(SUM)})', fontsize=9)
for a, m in zip(ax[0, 1:], mice):
    sm = SUM[SUM.mouse == m]; s = int(sm.sig.sum()); nt = len(sm)
    a.pie([s, nt - s], labels=[f'sig {s}', f'non {nt - s}'], colors=_sc, autopct='%1.0f%%', startangle=90)
    a.set_title(m, fontsize=9)
plt.suptitle(f'significant vs non-significant sessions  ({SIG_COL} < {SIG_THRESH})'); plt.tight_layout(); plt.show()

In [ ]:
# the same metrics, grouped by SIGNIFICANT vs NON-SIGNIFICANT session -- significant = p_all < 0.05
# (one definition everywhere), so in the p_all panel the significant dots all sit below the red line.
fig, axes = plt.subplots(1, 5, figsize=(18, 3.9))
for a, (key, lbl, ylim) in zip(axes, grid):
    dots_by_sig(a, SUM, key, lbl, ylim=ylim); a.set_title(lbl, fontsize=9)
    if key.startswith('p_'): a.axhline(0.05, ls='--', color='r', lw=.9)
axes[0].legend(handles=_mlegend, fontsize=6, title='mouse', loc='lower left')
plt.suptitle('metrics by significance (significant = p_all < 0.05) — dot = session (colour = mouse), bar = group mean')
plt.tight_layout(); plt.show()

### 4b — win-stay: what does he collect next?

For each collection, what he just picked up → is his **next** collection a reward? One value per session
(colour = mouse), grouped by what he just collected. If "after a reward" sits highest, he sticks to
reward once he's on a roll.

⚠️ **After a banishment the next collection is ALWAYS an escape** (he is in the shadow realm and must collect the unbanish ring), so *after banishment* is 0 by construction — the informative one is *after escape*: once he is back, does he go for a reward?

In [ ]:
def transitions(all_df):
    rows = []
    for (mouse, session), g in all_df.groupby(['mouse', 'session']):
        eff = g.sort_values('time_ms').effect.values
        if len(eff) < 2: continue
        prev, nxt_is_rew = eff[:-1], (eff[1:] == 'single_reward')
        for src in ['single_reward', 'banish', 'unbanish', 'timeout']:
            mask = prev == src
            rows.append(dict(mouse=mouse, session=session, prev=src,
                             p_next_reward=(float(nxt_is_rew[mask].mean()) if mask.any() else np.nan)))
    return pd.DataFrame(rows)
TR = transitions(ALL)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
dots_by_cat(ax, TR, 'prev', ['single_reward', 'banish', 'unbanish', 'timeout'], 'p_next_reward',
            'P(next collection = reward)',
            labels=['after\nreward', 'after\nbanishment', 'after\nescape', 'after\ntimeout'], ylim=(0, 1.02))
ax.set_title('win-stay: P(next = reward) by what he just collected'); ax.legend(handles=_mlegend, fontsize=7, title='mouse')
plt.tight_layout(); plt.show()

### 4b-ii — P(next is a reward): significant vs non-significant

The chance the next collection is a reward — overall, and split by what came just before (a reward =
win-stay, a negative icon = recovery) — compared between **significant** (p_all < 0.05) and
**non-significant** sessions.

In [ ]:
wg = [('reward_rate', 'P(reward) overall', (0, 1.02)),
      ('win_stay', 'P(next reward | after a reward)', (0, 1.02)),
      ('p_reward_after_bad', 'P(next reward | after a negative icon)', (0, 1.02))]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for a, (key, lbl, ylim) in zip(axes, wg):
    dots_by_sig(a, SUM, key, lbl, ylim=ylim); a.set_title(lbl, fontsize=9)
axes[0].legend(handles=_mlegend, fontsize=6, title='mouse', loc='lower left')
plt.suptitle('P(next is a reward): significant vs non-significant sessions (significant = p_all < 0.05)')
plt.tight_layout(); plt.show()
print('mean over sessions           significant   non-significant')
for key, lbl in [('reward_rate', 'P(reward) overall'), ('win_stay', 'P(reward | after reward)'),
                 ('p_reward_after_bad', 'P(reward | after negative)')]:
    sv = SUM[SUM.sig][key].mean(); nv = SUM[~SUM.sig][key].mean()
    print(f'  {lbl:28s} {sv:11.3f}   {nv:.3f}')

### 4c — rewards by multiplier level, per mouse

Grouped bar: for each multiplier (combo) level, the **proportion of that mouse's rewards** that landed
there — so the mice are comparable regardless of how many rewards each collected.

In [ ]:
mt = ALL[ALL.valence == 'positive'].dropna(subset=['multiplier']).copy(); mt['mult'] = mt.multiplier.astype(int)
piv = mt.groupby(['mouse', 'mult']).size().unstack(fill_value=0)
piv = piv.div(piv.sum(axis=1), axis=0)                    # proportion within each mouse
levels = sorted(mt['mult'].unique()); x = np.arange(len(levels)); w = 0.8 / max(len(mice), 1)
fig, ax = plt.subplots(figsize=(8, 4.2))
for i, m in enumerate(mice):
    ax.bar(x + i * w, [piv.loc[m, l] if (m in piv.index and l in piv.columns) else 0 for l in levels],
           w, color=MCOL[m], label=m)
ax.set_xticks(x + w * (len(mice) - 1) / 2); ax.set_xticklabels([f'x{l}' for l in levels])
ax.set_xlabel('reward multiplier (combo level)'); ax.set_ylabel('proportion of the mouse’s rewards')
ax.set_title('rewards by multiplier level, per mouse'); ax.legend(fontsize=7, title='mouse')
plt.tight_layout(); plt.show()

In [ ]:

# rewards by multiplier level: significant vs non-significant sessions (proportion of that group's rewards)
mt2 = mt.merge(SUM[['mouse', 'session', 'sig']], on=['mouse', 'session'], how='left')
levels2 = sorted(mt2['mult'].unique()); xx = np.arange(len(levels2)); w = 0.4
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for j, (sv, lbl, col) in enumerate([(True, 'significant', '#2ca25f'), (False, 'non-significant', '#bdbdbd')]):
    sub = mt2[mt2.sig == sv]
    prop = [float((sub['mult'] == l).mean()) if len(sub) else 0 for l in levels2]
    ax.bar(xx + (j - 0.5) * w, prop, w, color=col, label=lbl)
ax.set_xticks(xx); ax.set_xticklabels([f'x{l}' for l in levels2])
ax.set_xlabel('reward multiplier (combo level)'); ax.set_ylabel('proportion of the group’s rewards')
ax.set_title('rewards by multiplier: significant vs non-significant sessions'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 5 — timing, distance & path, by effect (session = dot)

Per session, the average **time** and **distance** to collect a reward vs a timeout vs a banishment, the
**path efficiency** of the approach (1 = beeline), and **how many** of each per session.

In [ ]:
effs = ['single_reward', 'banish', 'timeout']; elab = [mp.ELABEL[e] for e in effs]
fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
dots_by_cat(ax[0], PS, 'effect', effs, 'time_s', 'seconds', labels=elab); ax[0].set_title('avg time to collect')
dots_by_cat(ax[1], PS, 'effect', effs, 'dist', 'world units', labels=elab); ax[1].set_title('avg distance to collect')
dots_by_cat(ax[2], PS, 'effect', effs, 'pe', 'path efficiency (1=beeline)', labels=elab, ylim=(0, 1.02)); ax[2].set_title('path efficiency')
dots_by_cat(ax[3], PS, 'effect', effs, 'n', 'collections per session', labels=elab); ax[3].set_title('collections per session')
ax[0].legend(handles=_mlegend, fontsize=7, title='mouse', loc='upper right')
plt.suptitle('per-effect, per session (dot = session, colour = mouse; bar = across-session mean)')
plt.tight_layout(); plt.show()

In [ ]:

# per-effect metrics: significant (circle) vs non-significant (square) sessions
def eff_sig(ax, key, ylabel, ylim=None):
    for i, e in enumerate(effs):
        for sv, off, mk in [(True, -0.16, 'o'), (False, 0.16, 's')]:
            sub = PS[(PS.effect == e) & (PS.sig == sv)]
            for m in mice:
                v = sub[sub.mouse == m][key].dropna().values
                ax.scatter(np.full(len(v), i + off) + _rng.uniform(-.05, .05, len(v)), v, s=30, marker=mk,
                           color=MCOL[m], edgecolor='k', lw=.3, alpha=.8)
            allv = sub[key].dropna().values
            if len(allv): ax.scatter([i + off], [np.nanmean(allv)], marker='_', s=340, color='k', zorder=4)
    ax.set_xticks(range(len(effs))); ax.set_xticklabels(elab); ax.set_ylabel(ylabel, fontsize=8)
    if ylim: ax.set_ylim(*ylim)
fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
eff_sig(ax[0], 'time_s', 'seconds'); ax[0].set_title('avg time to collect')
eff_sig(ax[1], 'dist', 'world units'); ax[1].set_title('avg distance')
eff_sig(ax[2], 'pe', 'path efficiency', (0, 1.02)); ax[2].set_title('path efficiency')
eff_sig(ax[3], 'n', 'collections/session'); ax[3].set_title('collections per session')
_slg = [Line2D([0], [0], marker='o', ls='', color='0.4', label='significant'),
        Line2D([0], [0], marker='s', ls='', color='0.4', label='non-signif.')]
ax[0].legend(handles=_slg, fontsize=7, loc='upper right')
plt.suptitle('per-effect metrics: significant (circle) vs non-significant (square) sessions')
plt.tight_layout(); plt.show()


### 5b — time & distance between collections: significant vs non-significant

The single-session time / distance histograms, pooled ACROSS sessions and split by whether the session
was significant. Bars are **density-normalised** (the two groups have different collection counts), so
compare the SHAPE and the median (in the legend), not the bar height. One column per collected type.

In [ ]:
A = ALL.merge(SUM[['mouse', 'session', 'sig']], on=['mouse', 'session'], how='left')
A = A[A.dt_prev_ms.notna()]
effs3 = [('single_reward', 'reward'), ('timeout', 'timeout'), ('banish', 'banishment')]
fig, ax = plt.subplots(2, 3, figsize=(15, 7.5))
for j, (e, lbl) in enumerate(effs3):
    for sv, c, nm in [(True, '#2ca25f', 'significant'), (False, '#bdbdbd', 'non-sig')]:
        s = A[(A.effect == e) & (A.sig == sv)]
        if len(s):
            ax[0, j].hist(s.dt_prev_ms / 1000, bins=20, color=c, alpha=0.55, density=True,
                          label=f'{nm} (n={len(s)}, med {np.nanmedian(s.dt_prev_ms)/1000:.1f}s)')
            ax[1, j].hist(s.dist_prev.dropna(), bins=20, color=c, alpha=0.55, density=True,
                          label=f'{nm} (med {np.nanmedian(s.dist_prev):.0f} wu)')
    ax[0, j].set_title(f'{lbl}: time since previous'); ax[0, j].set_xlabel('time (s)'); ax[0, j].legend(fontsize=6)
    ax[1, j].set_title(f'{lbl}: distance from previous'); ax[1, j].set_xlabel('distance (wu)'); ax[1, j].legend(fontsize=6)
plt.suptitle('inter-collection TIME (top) & DISTANCE (bottom) by effect -- significant vs non-significant sessions (density)')
plt.tight_layout(); plt.show()

## 6 — occupancy exemplars: significant vs non-significant session

For each mouse, the icon-centred avatar occupancy (±3 s) in its **most-significant** session (lowest
p-all) vs its **least-significant** (highest p-all). `ICON` picks the effect (reward by default). Star =
the icon; colour = fraction of near-icon time.

In [ ]:
ICON = 'single_reward'                # 'single_reward' | 'banish' | 'timeout'
HALF = OCC_SESS['half']
fig, axes = plt.subplots(len(mice), 2, figsize=(8, 3.6 * len(mice)), squeeze=False)
for r, m in enumerate(mice):
    sig, nons = exemplars(m)
    for c, (row, tag) in enumerate([(sig, 'MOST significant'), (nons, 'LEAST significant')]):
        a = axes[r][c]
        if row is None: a.axis('off'); continue
        M = OCC_SESS[ICON].get((m, row.session))
        if M is None:
            a.text(0.5, 0.5, f'no {mp.ELABEL[ICON]}\ncollections', ha='center', va='center'); a.set_xticks([]); a.set_yticks([])
        else:
            im = a.imshow(M.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
            a.plot(0, 0, marker='*', ms=13, color='cyan', mec='k')
        a.set_title(f'{m}  {tag}\n{row.session}   p={row.p_min:.3f}', fontsize=8)
plt.suptitle(f'occupancy around the {mp.ELABEL[ICON]} icon — exemplar sessions per mouse', y=1.005)
plt.tight_layout(); plt.show()

### 6b — one animal, all effects

Zoom in on a single animal (`ANIMAL`): its occupancy around **all three** icon types (reward /
banishment / timeout), for its most- vs least-significant session. Does he sit around a reward
differently than around a negative icon he keeps hitting?

In [ ]:

ANIMAL = mice[0]                      # <-- pick the animal to look at
HALF = OCC_SESS['half']
sig, nons = exemplars(ANIMAL)
fig, axes = plt.subplots(2, 3, figsize=(13, 8.2), squeeze=False)
for r, (row, tag) in enumerate([(sig, 'MOST significant'), (nons, 'LEAST significant')]):
    for c, (e, lbl) in enumerate(mp.TYPES):
        a = axes[r][c]
        if row is None: a.axis('off'); continue
        M = OCC_SESS[e].get((ANIMAL, row.session))
        if M is None:
            a.text(0.5, 0.5, f'no {lbl}\ncollections', ha='center', va='center', transform=a.transAxes)
            a.set_xticks([]); a.set_yticks([])
        else:
            im = a.imshow(M.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
            fig.colorbar(im, ax=a, fraction=0.046, pad=0.04, label='fraction of near-icon time')
            a.plot(0, 0, marker='*', ms=12, color='cyan', mec='k')
        if r == 0: a.set_title(lbl, fontsize=11)
        if c == 0 and row is not None: a.set_ylabel(f'{tag}\n{row.session}\np={row.p_min:.3f}', fontsize=8)
plt.suptitle(f'{ANIMAL} — occupancy for all effects (rows = its significant vs non-significant session)', y=1.005)
plt.tight_layout(); plt.show()


## 7 — heading error + joystick fine movement (exemplars)

Both signals for the **same** session, stacked so you can compare them directly: heading error on top
(0 = facing the icon), joystick fine movement below it. Per mouse, its most- vs least-significant
session, the three effects overlaid.

In [ ]:

HG, JG = HEAD_SESS['grid'], JOY_SESS['grid']
fig, axes = plt.subplots(2 * len(mice), 2, figsize=(15, 2.7 * 2 * len(mice)), squeeze=False,
                         gridspec_kw={'hspace': 0.45, 'wspace': 0.16})
for r, m in enumerate(mice):
    sig, nons = exemplars(m)
    for c, (row, tag) in enumerate([(sig, 'MOST significant'), (nons, 'LEAST significant')]):
        ah, aj = axes[2 * r][c], axes[2 * r + 1][c]
        if row is None: ah.axis('off'); aj.axis('off'); continue
        for e, lbl in mp.TYPES:
            hc = HEAD_SESS[e].get((m, row.session)); jc = JOY_SESS[e].get((m, row.session))
            if hc is not None: ah.plot(HG, hc, color=mp.COLR[e], lw=2.2, label=lbl)
            if jc is not None: aj.plot(JG, jc, color=mp.COLR[e], lw=2.2)
        ah.axhline(90, ls=':', color='0.6'); ah.set_ylim(0, 180); ah.set_xticklabels([])
        ah.set_title(f'{m}  {tag}: {row.session}  (p={row.p_min:.3f})', fontsize=8)
        ah.set_ylabel('heading error (deg)', fontsize=9); aj.set_ylabel('joystick fine', fontsize=9)
        ah.grid(alpha=.25); aj.grid(alpha=.25)
        aj.axvline(0, ls=':', color='0.6'); aj.set_xlabel('time to collection (s)', fontsize=8)
axes[0][0].legend(fontsize=6)
plt.suptitle('heading (top) + joystick fine movement (bottom) — most vs least significant session per mouse', y=1.002)
plt.tight_layout(); plt.show()
